# Explainable AI (XAI) for Prediction Transparency

This notebook implements multiple XAI techniques:
1. **SHAP** (SHapley Additive exPlanations) - global and local explanations
2. **LIME** (Local Interpretable Model-agnostic Explanations)
3. **Feature Importance Analysis** - permutation and built-in
4. **Partial Dependence Plots**
5. **Decision Boundary Visualization**

## Section 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import re, warnings, time, os, json
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, cross_val_score)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from scipy.sparse import hstack, csr_matrix
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print('Libraries loaded.')
try:
    import shap
    HAS_SHAP = True
    print('SHAP available.')
except ImportError:
    HAS_SHAP = False
    print('shap not installed. pip install shap')

try:
    import lime
    import lime.lime_tabular
    HAS_LIME = True
    print('LIME available.')
except ImportError:
    HAS_LIME = False
    print('lime not installed. pip install lime')

from sklearn.inspection import permutation_importance, PartialDependenceDisplay

In [ ]:
df = pd.read_csv('../data/cumulative_ai_customer_communication_dataset.csv', low_memory=False)
df['issue_reported_at'] = pd.to_datetime(df['issue_reported_at'], errors='coerce', dayfirst=True)
df['issue_responded'] = pd.to_datetime(df['issue_responded'], errors='coerce', dayfirst=True)
df['target'] = (df['csat_score'] >= 4).astype(int)
print(f'Dataset: {df.shape[0]} rows, Target positive rate: {df["target"].mean()*100:.1f}%')

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def preprocess_text(text):
    if pd.isna(text) or not isinstance(text, str): return ''
    text = text.lower()
    text = text.encode('ascii','ignore').decode('ascii')
    text = re.sub(r'[^a-z\s]','',text)
    text = re.sub(r'\s+',' ',text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t)>1]
    return ' '.join(tokens)
print('Preprocessing text...')
df['cleaned_message'] = df['customer_message'].apply(preprocess_text)
print(f'Done. Non-empty: {(df["cleaned_message"]!="").sum()}')

In [ ]:
df['response_time_minutes'] = ((df['issue_responded']-df['issue_reported_at']).dt.total_seconds()/60).clip(lower=0).fillna(0)
df['issue_hour'] = df['issue_reported_at'].dt.hour.fillna(0).astype(int)
df['issue_day_of_week'] = df['issue_reported_at'].dt.dayofweek.fillna(0).astype(int)
for col,src in [('channel_encoded','channel_name'),('category_encoded','category'),
                ('subcategory_encoded','sub-category'),('shift_encoded','agent_shift')]:
    le=LabelEncoder(); df[col]=le.fit_transform(df[src].fillna('Unknown'))
tenure_map={'On Job Training':0,'0-30':1,'31-60':2,'61-90':3,'>90':4}
df['tenure_encoded']=df['tenure_bucket'].map(tenure_map).fillna(0).astype(int)
df['has_message']=(df['cleaned_message']!='').astype(int)
df['cleaned_word_count']=df['cleaned_message'].apply(lambda x:len(x.split()) if x else 0)
structured_features=['response_time_minutes','issue_hour','issue_day_of_week','channel_encoded',
    'category_encoded','subcategory_encoded','shift_encoded','tenure_encoded',
    'message_length','word_count','has_message','cleaned_word_count']
print(f'Structured features: {len(structured_features)}')

In [ ]:
X_structured = df[structured_features].fillna(0)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X_structured, y, test_size=0.2, random_state=42, stratify=y)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=5)
X_text_train = tfidf.fit_transform(df.loc[X_train.index,'cleaned_message'])
X_text_test = tfidf.transform(df.loc[X_test.index,'cleaned_message'])
X_combined_train = hstack([X_text_train, csr_matrix(X_train.values)])
X_combined_test = hstack([X_text_test, csr_matrix(X_test.values)])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg_count=(y_train==0).sum(); pos_count=(y_train==1).sum(); scale_weight=neg_count/pos_count
print(f'Train:{X_train.shape[0]}, Test:{X_test.shape[0]}, TF-IDF:{X_text_train.shape[1]}, Combined:{X_combined_train.shape[1]}')

## Section 2: Train Model for Explanation

In [ ]:
# Train XGBoost (primary model for explanation)
xgb_model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_weight, random_state=42, eval_metric='logloss', use_label_encoder=False)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:,1]
print(f'XGBoost: Acc={accuracy_score(y_test,y_pred)*100:.2f}%, F1={f1_score(y_test,y_pred)*100:.2f}%')

# Also train LR for coefficient interpretation
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train, y_train)
print(f'LR: Acc={accuracy_score(y_test,lr_model.predict(X_test))*100:.2f}%')

## Section 3: SHAP Analysis

In [ ]:
if HAS_SHAP:
    print('=== SHAP Global Feature Importance ===')
    explainer = shap.TreeExplainer(xgb_model)
    # Use subset for speed
    X_explain = X_test.iloc[:500]
    shap_values = explainer.shap_values(X_explain)

    # Summary plot
    fig, ax = plt.subplots(figsize=(10, 7))
    shap.summary_plot(shap_values, X_explain, feature_names=structured_features, show=False)
    plt.title('SHAP Feature Importance (Global)', fontsize=14)
    plt.tight_layout()
    plt.savefig('../models/shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Bar plot
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_values, X_explain, feature_names=structured_features, plot_type='bar', show=False)
    plt.title('SHAP Mean Absolute Values', fontsize=14)
    plt.tight_layout()
    plt.savefig('../models/shap_bar.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('SHAP not available - skipping')

In [ ]:
# SHAP local explanations (individual predictions)
if HAS_SHAP:
    print('=== SHAP Local Explanations (Sample Predictions) ===')
    # Explain specific predictions
    for i, idx in enumerate(X_explain.index[:3]):
        print(f'\nSample {i+1} (index={idx}):')
        print(f'  Actual: {y_test.loc[idx]}, Predicted: {xgb_model.predict(X_explain.loc[[idx]])[0]}')
        print(f'  Probability: {xgb_model.predict_proba(X_explain.loc[[idx]])[0,1]:.3f}')

    # Waterfall plot for first sample
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.waterfall_plot(shap.Explanation(
        values=shap_values[0], base_values=explainer.expected_value,
        data=X_explain.iloc[0], feature_names=structured_features), show=False)
    plt.title('SHAP Waterfall - Sample Prediction')
    plt.tight_layout()
    plt.savefig('../models/shap_waterfall.png', dpi=150, bbox_inches='tight')
    plt.show()

## Section 4: LIME Explanations

In [ ]:
if HAS_LIME:
    print('=== LIME Local Explanations ===')
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        X_train.values, feature_names=structured_features,
        class_names=['Negative', 'Positive'], mode='classification', random_state=42
    )

    # Explain 3 samples
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for i in range(3):
        idx = X_test.index[i]
        exp = lime_explainer.explain_instance(
            X_test.iloc[i].values, xgb_model.predict_proba, num_features=8
        )
        print(f'\nSample {i+1}: Actual={y_test.iloc[i]}, Pred={xgb_model.predict(X_test.iloc[[i]])[0]}')
        # Get feature contributions
        feat_imp = exp.as_list()
        features = [f[0] for f in feat_imp]
        weights = [f[1] for f in feat_imp]
        colors = ['green' if w > 0 else 'red' for w in weights]
        axes[i].barh(features, weights, color=colors)
        axes[i].set_title(f'Sample {i+1} (Actual={y_test.iloc[i]})')
        axes[i].axvline(x=0, color='black', linewidth=0.5)

    plt.suptitle('LIME Explanations - Individual Predictions', fontsize=14)
    plt.tight_layout()
    plt.savefig('../models/lime_explanations.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('LIME not available - skipping')

## Section 5: Permutation Importance

In [ ]:
# Permutation importance (model-agnostic)
print('=== Permutation Feature Importance ===')
perm_imp = permutation_importance(xgb_model, X_test, y_test, n_repeats=10, random_state=42, scoring='f1', n_jobs=-1)

perm_df = pd.DataFrame({
    'Feature': structured_features,
    'Importance Mean': perm_imp.importances_mean,
    'Importance Std': perm_imp.importances_std
}).sort_values('Importance Mean', ascending=False)
print(perm_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
perm_sorted = perm_df.sort_values('Importance Mean', ascending=True)
ax.barh(perm_sorted['Feature'], perm_sorted['Importance Mean'],
        xerr=perm_sorted['Importance Std'], color='steelblue', capsize=3)
ax.set_title('Permutation Feature Importance (XGBoost)', fontsize=12)
ax.set_xlabel('Mean F1 Score Decrease')
plt.tight_layout()
plt.savefig('../models/permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 6: Partial Dependence Plots

In [ ]:
# Partial dependence plots for top features
print('=== Partial Dependence Plots ===')
top_features_idx = perm_df.head(4)['Feature'].tolist()
top_idx = [structured_features.index(f) for f in top_features_idx]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, (feat_idx, feat_name) in enumerate(zip(top_idx, top_features_idx)):
    ax = axes[i//2, i%2]
    PartialDependenceDisplay.from_estimator(
        xgb_model, X_train, [feat_idx], feature_names=structured_features, ax=ax, n_jobs=-1
    )
    ax.set_title(f'PDP: {feat_name}')
plt.suptitle('Partial Dependence Plots (Top 4 Features)', fontsize=14)
plt.tight_layout()
plt.savefig('../models/partial_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 7: LR Coefficient Analysis

In [ ]:
# Logistic Regression coefficient interpretation
print('=== Logistic Regression Coefficients ===')
coef_df = pd.DataFrame({
    'Feature': structured_features,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if c > 0 else 'red' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.set_title('Logistic Regression Coefficients', fontsize=12)
ax.set_xlabel('Coefficient Value')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('../models/lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nPositive coefficients = increase P(Positive CSAT)')
print('Negative coefficients = increase P(Negative CSAT)')

## Section 8: Summary of Explainability Findings

In [ ]:
print('=== XAI SUMMARY ===')
print()
print('Key Findings:')
print('1. Feature Importance Ranking (consistent across methods):')
for i, row in perm_df.head(5).iterrows():
    print(f'   {row["Feature"]}: {row["Importance Mean"]:.4f}')
print()
print('2. Interpretability Methods Applied:')
methods = ['Built-in Feature Importance (XGBoost)','Permutation Importance','LR Coefficients']
if HAS_SHAP: methods.append('SHAP (Global + Local)')
if HAS_LIME: methods.append('LIME (Local)')
methods.append('Partial Dependence Plots')
for m in methods: print(f'   - {m}')
print()
print('3. All explanations saved to ../models/')
print('   These can be used for research papers, presentations, and stakeholder communication.')